# Session 16: Build a Support Ticket Agent

Reuse the adapter exported by the fine-tuning practice to classify tickets. Observe the model’s decisions and the results returned by the local guide lookup.

## 1. Upload the adapter ZIP

In Colab, choose a **T4 GPU**. Open the **Files** pane and upload `session16-ticket-adapter.zip` from the fine-tuning notebook. In local Jupyter, put it in the notebook’s working directory. The ZIP contains the adapter weights, its PEFT configuration and the training method.

In [ ]:
import os
import subprocess
import sys
from zipfile import ZipFile

import torch

ADAPTER_ZIP = "session16-ticket-adapter.zip"
if not os.path.isfile(ADAPTER_ZIP):
    raise FileNotFoundError(f"Upload {ADAPTER_ZIP} from the fine-tuning practice first.")

with ZipFile(ADAPTER_ZIP) as archive:
    method = archive.read("training_method.txt").decode("utf-8").strip()

if method not in {"lora", "qlora"}:
    raise ValueError("Use an adapter ZIP exported by the fine-tuning notebook.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if method == "qlora" and device.type != "cuda":
    raise RuntimeError("Select a Colab T4 GPU for this QLoRA adapter.")

packages = ["transformers==4.57.1", "accelerate==1.10.1", "peft==0.17.1"]
if method == "qlora":
    packages.append("bitsandbytes==0.48.1")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print(f"Adapter: {method} | Device: {device}")

## 2. Load Qwen and the adapter

Load the original instruct checkpoint once. PEFT attaches the trained adapter. QLoRA uses the same 4-bit loading settings as the fine-tuning practice.

In [ ]:
from tempfile import TemporaryDirectory

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

CHECKPOINT = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
dtype = torch.float16 if device.type == "cuda" else torch.float32
load_options = {"dtype": dtype}

if method == "qlora":
    load_options["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )
    load_options["device_map"] = {"": torch.cuda.current_device()}

base_model = AutoModelForCausalLM.from_pretrained(CHECKPOINT, **load_options)
if method == "lora":
    base_model.to(device)

with TemporaryDirectory() as adapter_dir, ZipFile(ADAPTER_ZIP) as archive:
    for filename in ("adapter_config.json", "adapter_model.safetensors"):
        archive.extract(filename, adapter_dir)
    model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

## 3. Load the support guides

The JSON file has one fictional guide per category. A local checkout uses the bundled file. In Colab the cell downloads the same file to the runtime, then reads it locally.

In [ ]:
import json
from urllib.request import urlretrieve

SESSION_DIR = "16-LLM-Fine-Tuning-and-Agents"
GUIDE_FILE = "sample_data/support_guides.json"
guide_paths = ("support_guides.json", GUIDE_FILE, f"../{GUIDE_FILE}", f"{SESSION_DIR}/{GUIDE_FILE}")
guide_path = next((path for path in guide_paths if os.path.isfile(path)), None)
if guide_path is None:
    guide_url = f"https://raw.githubusercontent.com/wlos/AI_SS/main/{SESSION_DIR}/{GUIDE_FILE}"
    guide_path, _ = urlretrieve(guide_url, "support_guides.json")

with open(guide_path, encoding="utf-8") as guide_file:
    support_guides = json.load(guide_file)

CATEGORIES = ("Account access", "Hardware", "Network", "Software")
print("Available guides:", ", ".join(support_guides))

## 4. Define the two model roles

Use the adapter for classification and temporarily disable it for decisions and replies. The classification tool uses the same prompt as the fine-tuning practice and checks that the output contains an allowed category.

In [ ]:
from contextlib import nullcontext

from transformers import GenerationConfig


@torch.inference_mode()
def generate_reply(prompt, use_adapter=False, max_new_tokens=64):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    generation = GenerationConfig(
        do_sample=False,
        max_new_tokens=max_new_tokens,
        eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    # Use the same model object for both roles.
    with nullcontext() if use_adapter else model.disable_adapter():
        tokens = model.generate(**inputs, generation_config=generation)
    reply = tokens[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()


def classify_ticket(ticket):
    prompt = (
        f'Classify this support ticket as one of {", ".join(CATEGORIES)}. '
        'Return only a JSON object with the key "category".\n'
        f"Ticket: {ticket}"
    )
    reply = generate_reply(prompt, use_adapter=True, max_new_tokens=32)
    try:
        result = json.loads(reply)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and set(result) == {"category"}:
        category = result["category"]
        if category in CATEGORIES:
            return category
    return None

## 5. Run the agent

Qwen decides whether the ticket needs clarification. If the ticket is clear, Python calls the classifier and looks up the validated category. The guide result goes back to Qwen to draft the reply. The returned trace shows each decision and tool result.

In [ ]:
def run_ticket_agent(ticket, guides):
    decision_prompt = (
        "Decide whether the support ticket has enough information to proceed. "
        "Return exactly ASK or CLASSIFY. "
        "For a vague login issue without the service name, return ASK. "
        "For a specific account, device, network or software issue, return CLASSIFY.\n"
        f"Ticket: {ticket}"
    )
    decision = generate_reply(decision_prompt, max_new_tokens=8).upper()
    needs_clarification = decision != "CLASSIFY"
    trace = {
        "decision": decision,
        "category": None,
        "guide": None,
        "needs_clarification": needs_clarification,
    }
    if needs_clarification:
        trace["reply"] = "Which service or device is affected, and what happened?"
        return trace

    category = classify_ticket(ticket)
    trace["category"] = category
    if category is None:
        trace["reply"] = "I could not classify this ticket. Please contact the service desk."
        return trace

    guide = guides.get(category)
    trace["guide"] = guide
    if guide is None:
        trace["reply"] = "I could not find a guide for this issue. Please contact the service desk."
        return trace

    trace["reply"] = generate_reply(
        f"Write a short support reply using only the guide below.\nTicket: {ticket}\nGuide: {guide}"
    )
    return trace

## 6. Chat with the ticket agent

Run the cell and type a support request. Try **I can't log in.** If the agent asks for more detail, reply **My student portal account is locked.** The chat combines both messages into one ticket. After a final reply, the next message starts a new ticket.

Type `/exit` to end the chat. You can inspect `last_trace` after the chat to see the latest decision, category and guide.

In [ ]:
ticket_context = []
last_trace = None
print("Ticket support chat. Type /exit to stop.")

while True:
    message = input("You: ").strip()
    if message.casefold() == "/exit":
        break
    if not message:
        continue

    ticket_context.append(message)
    last_trace = run_ticket_agent(" ".join(ticket_context), support_guides)
    print(f"Agent: {last_trace['reply']}")

    # Keep earlier details while the agent is waiting for clarification.
    if not last_trace["needs_clarification"]:
        ticket_context.clear()